In [1]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

import MDAnalysis as mda
from MDAnalysis.analysis import distances
import networkx as nx

from polym_analysis import *
from gromacs_topology import *

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [ ]:
# CPK color scheme
cpk_colors = {
    'H': '#FFFFFF', 'C': '#000000', 'N': '#3050F8', 'O': '#FF0D0D',
    'F': '#90E050', 'P': '#FF8000', 'S': '#FFFF30', 'Cl': '#1FF01F',
    'Br': '#A62929', 'I': '#940094', 'Na': '#AB5CF2', 'Mg': '#8AFF00',
    'Ca': '#3DFF00', 'Fe': '#E06633', 'Zn': '#7D80B0'
}

def draw_graph(G: nx.Graph, pos=None, show_atoms=True):
    '''
    Draw a NetworkX graph with nodes colored by element type.

    Parameters
    ----------
    G: NetworkX Graph
        The graph to draw.
    pos: dict, optional
        A dictionary mapping node indices to positions. If None, a spring layout will be used.
    '''

    # Extract node colors and labels
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_sizes = [75*G.nodes[node]['atom'].mass for node in G.nodes()]

    # Create the plot
    plt.figure(figsize=(8, 6))
    if pos is None:
        pos = nx.spring_layout(G, k=1, iterations=50)

    # Draw edges first
    nx.draw_networkx_edges(G, pos, alpha=0.6, width=2)

    # Draw nodes with colors
    if show_atoms:
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors='k', 
                            node_size=node_sizes, alpha=0.9)

    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [3]:
def random_amides(polym, num, seed=12345):
    '''Randomly select num of amide bonds to break'''

    sel = polym.universe.select_atoms('type n')
    n = len(sel)
    rng = np.random.default_rng(seed=seed)
    rand_idx = rng.choice(np.arange(0,n), size=num, replace=False)
    
    return sel[rand_idx]

### Test out breaking bonds

In [ ]:
path = '/Users/nate_schwindt/Projects/polymer_membrane/polyamide_XAS_MD/Na/NaCl_polymer/50pct_deprotonated/OPC3/placed_Na/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'prod.gro', frmt='GRO', tpr_file=path+'prod.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

poly.plot_membrane(poly.polymer)

In [ ]:
pa = poly.polymer.residues[3].atoms
pa_graph = build_graph_from_atoms(pa)

print('Before breaking bonds:', nx.number_connected_components(pa_graph))

pos = {n: (d['atom'].position[0], d['atom'].position[2]) for n, d in pa_graph.nodes(data=True)}
draw_graph(pa_graph, pos=pos, show_atoms=False)

In [ ]:
xlink_n = pa.select_atoms('type n')
to_break = [b for b in xlink_n[4].bonds if 'n' in b.type and 'c' in b.type][0]
print('Before breaking bonds:', nx.number_connected_components(pa_graph), pa.bonds)
pa.universe.delete_bonds([to_break])

pa_graph = build_graph_from_atoms(pa)
print('After breaking bonds:', nx.number_connected_components(pa_graph), pa.bonds)

pos = {n: (d['atom'].position[0], d['atom'].position[2]) for n, d in pa_graph.nodes(data=True)}
draw_graph(pa_graph, pos=pos, show_atoms=False)

### Break bonds

In [67]:
path = '/Users/nate_schwindt/Projects/polymer_membrane/polyamide_XAS_MD/Na/NaCl_polymer/50pct_deprotonated/OPC3/placed_Na/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'prod.gro', frmt='GRO', tpr_file=path+'prod.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

Initializing polyamide membrane for analysis...
PA membrane consists of 716 MPD monomers, 444 TMC monomers, 6120 waters for a total of 36086 atoms
Finished loading PA membrane for analysis!



In [68]:
# determine crosslinks to break, waters to remove, and positions to add Cl and OH
target_crosslink_pct = 0.8
radius_for_water_search = 6  # angstroms
rng_seed = 3

n_xlink = len(poly.universe.select_atoms('type n'))
n_possible = len(poly.universe.select_atoms('type n nh'))
n_to_break = n_xlink - int(target_crosslink_pct * n_possible)

poly.create_polymer_graphs()
n_components = [nx.number_connected_components(g) for g in poly.polymer_graphs]

print(f'Going from {100 * n_xlink / n_possible:.1f}% crosslinking to {100 * (n_xlink - n_to_break) / n_possible:.1f}% crosslinking by breaking {n_to_break} amide bonds')
                                  
print('\nNumber of bonds before breaking crosslinks:', len(poly.universe.bonds))
print('\nNumber of polymer components before breaking crosslinks:', sum(n_components))
print('Number of water molecules before breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))

# delete amide bonds
amide_Ns = random_amides(poly, n_to_break, seed=rng_seed)
amide_Cs = poly.universe.select_atoms('(type c) and (bonded group amide_Ns)', amide_Ns=amide_Ns)
to_break = [b for b in amide_Ns.bonds if 'n' in b.type and 'c' in b.type]

print(f'Breaking {len(to_break)} bonds')
poly.universe.delete_bonds(to_break)

# locate nearby waters to replace with Cl
to_delete = []

print(f'Trying to delete {n_to_break} waters and adding {n_to_break} Cl atoms')
Cl_positions = np.zeros((n_to_break, 3))
for i,n in enumerate(amide_Ns):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {n.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide N atom index', n.index)
    
    # save the positions of the waters to be replaced
    Cl_positions[i] = my_waters[0].position

    # remove the water molecule
    to_delete.append(my_waters[0].residue)
    
# locate nearby waters to replace with OH
print(f'Trying to delete {n_to_break} waters and adding {n_to_break} O atoms and {n_to_break} H atoms')
O_positions = np.zeros((n_to_break, 3))
H_positions = np.zeros((int(n_to_break/2), 3))
for i,c in enumerate(amide_Cs):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {c.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide C atom index', c.index)
    
    # save the positions of the waters to be replaced
    O_positions[i] = my_waters[0].position

    if i < int(n_to_break/2):
        H_positions[i] = my_waters[0].bonded_atoms[0].position  # for 50% deprotonated, we want to add O- for half and OH for half

    # remove the water molecule
    to_delete.append(my_waters[0].residue)

to_delete = list(set(to_delete))  # remove duplicates

Going from 93.2% crosslinking to 80.0% crosslinking by breaking 189 amide bonds

Number of bonds before breaking crosslinks: 31211

Number of polymer components before breaking crosslinks: 14
Number of water molecules before breaking crosslinks: 6120
Breaking 189 bonds
Trying to delete 189 waters and adding 189 Cl atoms
Trying to delete 189 waters and adding 189 O atoms and 189 H atoms


In [69]:
# Now delete all waters at once
print(f'Actually deleting {len(to_delete)} water molecules')
for i,water_res in enumerate(to_delete):
    if i == 0:
        all_waters_to_delete = water_res.atoms
    else:
        all_waters_to_delete += water_res.atoms

poly.universe.delete_bonds([b for b in all_waters_to_delete.bonds]) # also all bonds associated with deleted waters
poly.universe.atoms = poly.universe.atoms.subtract(all_waters_to_delete)

poly.polymer = poly.universe.select_atoms(f'not type {poly.ow_type} {poly.hw_type} {poly.anion_type} {poly.cation_type}') # reselect polymer atoms after deleting waters
poly.create_polymer_graphs()
n_components = [nx.number_connected_components(g) for g in poly.polymer_graphs]

# print('\nNumber of bonds after breaking crosslinks:', len(poly.universe.bonds))
print('\nNumber of polymer chains after breaking crosslinks:', sum(n_components))
print('Number of water molecules after breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))

Actually deleting 206 water molecules

Number of polymer chains after breaking crosslinks: 117
Number of water molecules after breaking crosslinks: 5914


In [70]:
# Reassign residue indices based on connected components
atom_to_new_resindex = {}
current_resid = 1
n_new_residues = 0

for residue in poly.polymer.residues:
    res_atoms = residue.atoms
    res_graph = build_graph_from_atoms(res_atoms)
    
    # Get connected components for this residue
    components = list(nx.connected_components(res_graph))
    
    for component in components:
        # Assign all atoms in this component to the same new residue
        for atom_idx in component:
            atom_to_new_resindex[atom_idx] = n_new_residues
        n_new_residues += 1

# Create a new empty universe with correct number of residues
n_atoms = len(poly.polymer.atoms)
new_polymer_u = mda.Universe.empty(
    n_atoms=n_atoms,
    n_residues=n_new_residues,
    atom_resindex=[atom_to_new_resindex[atom.index] for atom in poly.polymer.atoms],
    trajectory=True
)

# Copy over topology attributes
new_polymer_u.add_TopologyAttr('name', [atom.name for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('type', [atom.type for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('element', [atom.element for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('mass', [atom.mass for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('resname', [f'PA{i}' for i in range(1, n_new_residues + 1)])
new_polymer_u.add_TopologyAttr('resid', [i for i in range(1, n_new_residues + 1)])

# Copy positions and bonds
new_polymer_u.atoms.positions = poly.polymer.atoms.positions
bond_indices = [(b.atoms[0].index, b.atoms[1].index) for b in poly.polymer.bonds]
new_polymer_u.add_TopologyAttr('bonds', bond_indices)

# Merge with non-polymer atoms
non_polymer = poly.universe.atoms - poly.polymer.atoms
merged_u = mda.Merge(new_polymer_u.atoms, non_polymer)
merged_u.dimensions = poly.universe.dimensions

# Reassign resids to be sequential
# Polymer residues: 1 to n_new_residues
# Non-polymer residues: n_new_residues+1 onwards
n_polymer_atoms = len(new_polymer_u.atoms)
non_polymer_start_idx = n_new_residues + 1

# Get the non-polymer residues and renumber them
for i, res in enumerate(merged_u.residues[n_new_residues:], start=non_polymer_start_idx):
    for atom in res.atoms:
        atom.residue.resid = i

# Replace the universe
poly.universe = merged_u

print(f'Number of new polymer residues: {n_new_residues}')
print(f'Number of polymer chains after breaking crosslinks: {sum(n_components)}')

poly.universe.atoms.write('broken_bonds.gro')

IndexError: index 35469 is out of bounds for axis 0 with size 35468

In [ ]:
poly.polymer = poly.universe.select_atoms(f'not type {poly.ow_type} {poly.hw_type} {poly.anion_type} {poly.cation_type}')
poly.create_polymer_graphs()

for g in poly.polymer_graphs:
    pos = {n: (d['atom'].position[0], d['atom'].position[2]) for n, d in g.nodes(data=True)}
    draw_graph(g, pos=pos, show_atoms=False)

In [ ]:
# Build residue index arrays for new atoms
# Cl atoms go to amide_N residues, O and H atoms go to amide_C residues
new_atom_resindices = []

# Get the new resindices for the broken amide atoms
for n in amide_Ns:
    new_atom_resindices.append(atom_to_new_resindex[n.index])  # Cl atoms

for c in amide_Cs:
    new_atom_resindices.append(atom_to_new_resindex[c.index])  # O atoms

for i, c in enumerate(amide_Cs):
    if i < len(H_positions):
        new_atom_resindices.append(atom_to_new_resindex[c.index])  # H atoms

# Create empty universe for new atoms
n_added = len(Cl_positions) + len(O_positions) + len(H_positions)
new_atoms_u = mda.Universe.empty(
    n_atoms=n_added,
    n_residues=len(poly.universe.residues),  # Don't create new residues, but use the existing residues
    atom_resindex=new_atom_resindices,
    trajectory=True
)

# Set atom attributes
new_atoms_u.add_TopologyAttr('name', ['Cl'] * len(Cl_positions) + ['O'] * len(O_positions) + ['H'] * len(H_positions))
new_atoms_u.add_TopologyAttr('type', ['Cl'] * len(Cl_positions) + ['O'] * len(O_positions) + ['H'] * len(H_positions))
new_atoms_u.add_TopologyAttr('element', ['Cl'] * len(Cl_positions) + ['O'] * len(O_positions) + ['H'] * len(H_positions))
new_atoms_u.add_TopologyAttr('resname', [res.resname for res in poly.universe.residues])
new_atoms_u.add_TopologyAttr('resid', [res.resid for res in poly.universe.residues])

# Set positions
new_atoms_u.atoms.positions = np.vstack([Cl_positions, O_positions, H_positions])

# Merge with existing universe
final_u = mda.Merge(poly.universe.atoms, new_atoms_u.atoms)
final_u.dimensions = poly.universe.dimensions

# Replace the universe
poly.universe = final_u

print(f'Added {n_added} new atoms to existing residues')
print(f'Total atoms: {len(poly.universe.atoms)}')

poly.universe.atoms.write('chlorinated.gro') # may need to manually move the new polymer atoms above water and ions

In [ ]:
def get_bond(top, atomtype1, atomtype2):
    '''Get the bond between two atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2]

    for b in top.bonds:
        my_types = [a.type for a in b.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return b
    return None

def get_angle(top, atomtype1, atomtype2, atomtype3):
    '''Get the angle among three atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2, atomtype3]

    for a in top.angles:
        my_types = [a.type for a in a.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return a
    return None

def get_dihedral(top, atomtype1, atomtype2, atomtype3, atomtype4, is_improper=False):
    '''Get the dihedral among four atom types in a topology. Should specify if this dihedral is improper.'''

    atomtypes = [atomtype1, atomtype2, atomtype3, atomtype4]
    improper_types = [2, 4] # either harmonic (2) or periodic (4) improper

    for d in top.dihedrals:
        my_types = [a.type for a in d.atoms]
        if Counter(my_types) == Counter(atomtypes):
            if is_improper and d.type in improper_types:
                return d
            
            if not is_improper and d.type not in improper_types:
                return d
            
    return None

In [ ]:
param_top = GromacsTopology('2MPD_2TMC_1P_Cl_converted.top')

# test bonding retrieval functions
print('Testing bond retrieval:')
b = get_bond(param_top, 'ca', 'ca')
print([a.type for a in b.atoms])

b1 = get_bond(param_top, 'nh', 'cl')
b2 = get_bond(param_top, 'cl', 'nh')
print(b1 == b2)

b = get_bond(param_top, 'c', 'g')
print(b)

print('\nTesting angle retrieval:')
ang = get_angle(param_top, 'ha', 'ca', 'ca')
print([a.type for a in ang.atoms])

ang1 = get_angle(param_top, 'cl', 'nh', 'ca')
ang2 = get_angle(param_top, 'ca', 'nh', 'cl')
print(ang1 == ang2)

print('\nTesting dihedral retrieval:')
dih = get_dihedral(param_top, 'ca', 'ca', 'ca', 'ha')
print([a.type for a in dih.atoms])

dih1 = get_dihedral(param_top, 'cl', 'nh', 'ca', 'ca')
dih2 = get_dihedral(param_top, 'ca', 'ca', 'nh', 'cl')
print(dih1 == dih2)

dih1 = get_dihedral(param_top, 'ca', 'ca', 'ca', 'ha', is_improper=True)
print([a.type for a in dih1.atoms])

dih1 = get_dihedral(param_top, 'ca', 'ca', 'ca', 'ha', is_improper=True)
dih2 = get_dihedral(param_top, 'ca', 'ca', 'ca', 'ha', is_improper=False)
print(dih1 == dih2)

In [ ]:
# Load the topology
top = GromacsTopology('./PA_ions.top', verbose=True)

# 1. Delete broken amide bonds
for atom in top.atoms[amide_Ns.indices]:
    molecule = atom.molecule
    print('\nMolecule:', molecule)
    print(f'\tBonds before deletion: {len(top.bonds)}')
    print(f'\tAngles before deletion: {len(top.angles)}')
    print(f'\tDihedrals before deletion: {len(top.dihedrals)}')

    # Remove amide bonds, angles, and dihedrals
    for bond in atom.bonds:
        a_types = [a.type for a in bond.atoms]
        if 'c' in a_types and 'n' in a_types:
            molecule.bonds.remove(bond)

    for angle in atom.angles:
        a_types = [a.type for a in angle.atoms]
        if 'c' in a_types and 'n' in a_types:
            molecule.angles.remove(angle)

    for dih in atom.dihedrals:
        a_types = [a.type for a in dih.atoms]
        if 'c' in a_types and 'n' in a_types:
            molecule.dihedrals.remove(dih)

    print(f'\tBonds after deletion: {len(top.bonds)}')
    print(f'\tAngles after deletion: {len(top.angles)}')
    print(f'\tDihedrals after deletion: {len(top.dihedrals)}')

In [ ]:
# 2. Get bond parameters for new bonds from your parameter topology
param_gro = PolymAnalysis('2MPD_2TMC_1P_Cl_box.gro', frmt='GRO', tpr_file='2MPD_2TMC_1P_Cl_converted.tpr', 
                            xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n, cl_type=cl_type, 
                            ow_type=ow_type, hw_type=hw_type)
param_top = GromacsTopology('2MPD_2TMC_1P_Cl_converted.top')

cl = param_gro.universe.select_atoms('type cl')
cl_atom = param_top.atoms[cl.indices[0]]
nh_type = param_top.atomtypes['nh'] # broken amide N changes to type nh (amine N)

# bond parameters
nh_cl_bond = get_bond(param_top, 'nh', 'cl')
c_oh_bond = get_bond(param_top, 'c', 'oh')
oh_ho_bond = get_bond(param_top, 'oh', 'ho')
c_o_bond = get_bond(param_top, 'c', 'o')

print(f'\nnh-cl bond parameters: {[p for p in nh_cl_bond._params]}')
print(f'c-oh bond parameters: {[p for p in c_oh_bond._params]}')
print(f'oh-ho bond parameters: {[p for p in oh_ho_bond._params]}')
print(f'c-o bond parameters: {[p for p in c_o_bond._params]}')

# angle parameters
ca_nh_cl_angle = get_angle(param_top, 'ca', 'nh', 'cl')
hn_nh_cl_angle = get_angle(param_top, 'hn', 'nh', 'cl')
ca_c_oh_angle = get_angle(param_top, 'ca', 'c', 'oh')
c_oh_ho_angle = get_angle(param_top, 'c', 'oh', 'ho')
o_c_oh_angle = get_angle(param_top, 'o', 'c', 'oh')
o_c_o_angle = get_angle(param_top, 'o', 'c', 'o')
ca_c_o_angle = get_angle(param_top, 'ca', 'c', 'o')

print(f'\nca-nh-cl angle parameters: {[p for p in ca_nh_cl_angle._params]}')
print(f'hn-nh-cl angle parameters: {[p for p in hn_nh_cl_angle._params]}')
print(f'ca-c-oh angle parameters: {[p for p in ca_c_oh_angle._params]}')
print(f'c-oh-ho angle parameters: {[p for p in c_oh_ho_angle._params]}')
print(f'o-c-oh angle parameters: {[p for p in o_c_oh_angle._params]}')
print(f'o-c-o angle parameters: {[p for p in o_c_o_angle._params]}')

# dihedral parameters
# ca_nh_cl_hn_imp = get_dihedral(param_top, 'ca', 'nh', 'cl', 'hn', is_improper=True) # this dihedral does not actually appear in the moltemplate GAFF topology
ca_ca_nh_cl_dih = get_dihedral(param_top, 'ca', 'ca', 'nh', 'cl')
ca_c_o_oh_imp = get_dihedral(param_top, 'ca', 'c', 'o', 'oh', is_improper=True)
o_c_oh_ho_dih = get_dihedral(param_top, 'o', 'c', 'oh', 'ho')
ca_c_oh_ho_dih = get_dihedral(param_top, 'ca', 'c', 'oh', 'ho')
ca_ca_c_oh_dih = get_dihedral(param_top, 'ca', 'ca', 'c', 'oh')
ca_c_o_o_imp = get_dihedral(param_top, 'ca', 'c', 'o', 'o', is_improper=True)
ca_ca_c_o_dih = get_dihedral(param_top, 'ca', 'ca', 'c', 'o')

# print(f'\nca-nh-cl-hn improper parameters: {[p for p in ca_nh_cl_hn_imp._params]}')
print(f'\nca-ca-nh-cl dihedral parameters: {[p for p in ca_ca_nh_cl_dih._params]}')
print(f'ca-c-o-oh improper parameters: {[p for p in ca_c_o_oh_imp._params]}')
print(f'o-c-oh-ho dihedral parameters: {[p for p in o_c_oh_ho_dih._params]}')
print(f'ca-c-oh-ho dihedral parameters: {[p for p in ca_c_oh_ho_dih._params]}')
print(f'ca-ca-c-oh dihedral parameters: {[p for p in ca_ca_c_oh_dih._params]}')
print(f'ca-c-o-o improper parameters: {[p for p in ca_c_o_o_imp._params]}')
print(f'ca-ca-c-o dihedral parameters: {[p for p in ca_ca_c_o_dih._params]}')

In [ ]:
# 3. Add new Cl, O, H atoms to topology
n_cl_added = len(Cl_positions)
n_o_added = len(O_positions)
n_h_added = len(H_positions)

n_new_atoms = n_cl_added + n_o_added + n_h_added
print(f'Added {n_cl_added} Cl atoms, {n_o_added} O atoms, and {n_h_added} H atoms')

# Track indices for accessing atoms later
added_atoms_indices = poly.universe.atoms[-n_added:].indices

added_Cl_indices = added_atoms_indices[:n_cl_added]
added_O_indices = added_atoms_indices[n_cl_added:n_cl_added+n_o_added]
added_H_indices = added_atoms_indices[n_cl_added+n_o_added:n_cl_added+n_o_added+n_h_added]

print(f'Added Cl atom indices from {added_Cl_indices[0]} to {added_Cl_indices[-1]}')
print(f'Added O atom indices from {added_O_indices[0]} to {added_O_indices[-1]}')
print(f'Added H atom indices from {added_H_indices[0]} to {added_H_indices[-1]}')

In [ ]:
# create new list of molecules for the topology
new_molecules = []

# first for the polymer
for res in poly.polymer.residues:
    resname = res.resname
    n_mol = 1 # all residues are unique

    mol = Molecule(resname, n_mol)
    new_molecules.append(mol)

# water molecules
poly.universe.select_atoms('resname SOL')

# cations, anions, and extra cations    

In [ ]:
top.molecules[14].n_mols

In [ ]:
poly.universe.select_atoms('resname SOL').residues